# 🚀 LAB GUIDE — PRODUCTION-GRADE GRAPHRAG VS FLAT RAG — OPENROUTER EDITION

**Thời lượng:** 120 phút  
**Môi trường:** Google Colab (T4 GPU khuyến nghị) + Neo4j AuraDB  
**Dữ liệu:** HackerNoon Tech Company News Data Dump (bản thu gọn do giảng viên cung cấp)  
**Công cụ:** Học viên được dùng AI Coding Agent, nhưng phải tự thiết kế, kiểm thử và giải thích logic.

## 🎯 Mục tiêu
1. Xây dựng Hybrid GraphRAG end-to-end.
2. Xử lý Coreference Resolution, Entity Resolution và Super-node Mitigation.
3. Bulk insert bằng `UNWIND`, không insert từng row.
4. So sánh Flat RAG và GraphRAG bằng Golden Dataset + LLM-as-a-Judge.
5. Đo quality, latency và token usage.
6. Giải thích kiến trúc và failure modes.

> Notebook là **reference lab guide**: có code khung chạy được nhưng vẫn yêu cầu học viên thay prompt/threshold/retrieval policy và thuyết minh lựa chọn.

## ⏳ Timeline

| Phút | Nội dung |
|---|---|
| 00–15 | Setup, load, dedup, chunk, coreference |
| 15–45 | NER/RE, entity resolution, Neo4j bulk insert |
| 45–75 | Flat RAG, graph traversal, hybrid retrieval |
| 75–105 | Golden Dataset, LLM-as-a-Judge, comparison |
| 105–120 | Failure-mode tests, bonus, export, thuyết minh |

### Scale guard
Trong lab 2 giờ, không nên gửi toàn bộ 350MB qua LLM. Mặc định dùng subset:
- `LAB_MAX_ARTICLES = 1500`
- `LAB_MAX_CHUNKS = 3000`
- `EXTRACTION_MAX_CHUNKS = 400`

Kiến trúc phải scale được; volume trong giờ lab chỉ dùng để chứng minh pipeline.

# PHẦN 1 — SETUP & PREPROCESSING

### Secrets trên Colab
Tạo:
- `NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD`
- `OPENROUTER_API_KEY`
- `OPENROUTER_MODEL` — model chính cho coreference, extraction và answer generation
- `OPENROUTER_JUDGE_MODEL` — optional; nếu bỏ trống sẽ dùng `OPENROUTER_MODEL`
- `HF_TOKEN` để stream dataset từ Hugging Face
- optional: `OPENROUTER_HTTP_REFERER`, `OPENROUTER_APP_TITLE`

Ví dụ model slug: dùng đúng slug đang hiển thị trong OpenRouter Models, dạng `provider/model-name`.
Notebook dùng OpenAI-compatible endpoint `https://openrouter.ai/api/v1`.

Không hard-code API key vào notebook nộp bài.

In [23]:
#@title 1.1 — Install
%pip -q install neo4j pandas numpy pyarrow sentence-transformers faiss-cpu openai tqdm networkx spacy datasets datasketch langchain-community llama-index


In [24]:
#@title 1.2 — Imports & config
import os, re, json, time, random, hashlib, unicodedata
from pathlib import Path
from collections import defaultdict, Counter, deque
from difflib import SequenceMatcher

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from neo4j import GraphDatabase
from sentence_transformers import SentenceTransformer
import faiss

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.max_colwidth", 120)

def get_secret(name, default=None):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value is not None:
            return value
    except Exception:
        pass
    return os.environ.get(name, default)

# Keep infrastructure credentials out of the submitted notebook.
NEO4J_URI = get_secret("NEO4J_URI", "")
NEO4J_USER = get_secret("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = get_secret("NEO4J_PASSWORD", "")
NEO4J_DATABASE = get_secret("NEO4J_DATABASE", "neo4j")

OPENROUTER_API_KEY = get_secret("OPENAI_API_KEY", "")
OPENROUTER_MODEL = get_secret("OPENROUTER_MODEL", "poolside/laguna-xs-2.1")
OPENROUTER_JUDGE_MODEL = get_secret("OPENROUTER_JUDGE_MODEL", OPENROUTER_MODEL or "poolside/laguna-xs-2.1")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OPENROUTER_HTTP_REFERER = get_secret("OPENROUTER_HTTP_REFERER", "")
OPENROUTER_APP_TITLE = get_secret("OPENROUTER_APP_TITLE", "GraphRAG vs Flat RAG Production Lab")
HF_TOKEN = get_secret("HF_TOKEN", "")

DATA_PATH = "/content/hackernoon_subset.csv"
LAB_MAX_ARTICLES = 1500
LAB_MAX_CHUNKS = 3000
EXTRACTION_MAX_CHUNKS = 400
CHUNK_WORDS = 220
CHUNK_OVERLAP_WORDS = 40

# Entity-resolution / retrieval parameters are centralized for reproducibility.
ENTITY_VECTOR_THRESHOLD = 0.90
ENTITY_TOP_K = 5
SEED_FUZZY_THRESHOLD = 0.80


## 1.3 — Download HackerNoon Dataset bằng Hugging Face Streaming

Cell dưới đây stream trực tiếp dataset **`HackerNoon/tech-company-news-data-dump`** và ghi dần ra CSV, nên không cần tải toàn bộ dataset vào RAM.

### Hai cơ chế giới hạn

- `LIMIT_ROWS`: số dòng tối đa.
- `LIMIT_MB`: dung lượng file tối đa.
- `PRIORITIZE_MB = True`: ưu tiên dừng theo dung lượng MB.
- `PRIORITIZE_MB = False`: thanh tiến trình theo số dòng, nhưng **vẫn giữ hard-stop `LIMIT_ROWS`**.

### Lưu ý

- Đặt `HF_TOKEN` trong **Colab Secrets**, không hard-code token vào notebook.
- Nếu dataset yêu cầu quyền truy cập/gated access, hãy mở trang dataset trên Hugging Face và hoàn tất bước **Agree/Request access** trước.
- Sau khi cell hoàn tất, `DATA_PATH` mặc định đã trỏ tới `/content/hackernoon_subset.csv`, nên cell loader kế tiếp có thể chạy trực tiếp.

In [25]:
#@title 1.3 — Stream HackerNoon dataset -> CSV
import csv
import os
from datasets import load_dataset
from tqdm.auto import tqdm

DATASET_NAME = "HackerNoon/tech-company-news-data-dump"
OUTPUT_CSV = "/content/hackernoon_subset.csv"

# Giới hạn cho bản lab. Có thể tăng sau buổi học.
LIMIT_ROWS = 500_000
LIMIT_MB = 150

# True  -> progress/dừng ưu tiên theo MB
# False -> progress theo rows; vẫn có hard-stop LIMIT_ROWS
PRIORITIZE_MB = True

# Đọc từ Colab Secrets qua get_secret() ở cell config.
if not HF_TOKEN:
    raise ValueError(
        "Thiếu HF_TOKEN. Hãy thêm Hugging Face Access Token vào Colab Secrets với tên HF_TOKEN."
    )

print("Đang kết nối luồng dữ liệu (streaming)...")

try:
    dataset = load_dataset(
        DATASET_NAME,
        split="train",
        streaming=True,
        token=HF_TOKEN,
    )
    iterator = iter(dataset)

    first_row = next(iterator)
    headers = list(first_row.keys())

    print(f"Đang ghi dữ liệu vào: {OUTPUT_CSV}")

    rows_written = 0
    total_progress = LIMIT_MB if PRIORITIZE_MB else LIMIT_ROWS
    unit_progress = "MB" if PRIORITIZE_MB else "row"

    with open(OUTPUT_CSV, mode="w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=headers, extrasaction="ignore")
        writer.writeheader()
        writer.writerow(first_row)
        rows_written += 1

        # Flush để kích thước file phản ánh dữ liệu vừa ghi.
        f.flush()
        file_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)

        with tqdm(
            total=total_progress,
            desc=f"Đang tải ({unit_progress})",
            unit=unit_progress,
        ) as pbar:
            if PRIORITIZE_MB:
                pbar.n = min(file_size_mb, LIMIT_MB)
                pbar.refresh()
            else:
                pbar.update(1)

            for row in iterator:
                writer.writerow(row)
                rows_written += 1

                # Kiểm tra dung lượng định kỳ để giảm overhead I/O.
                # Khi gần LIMIT_MB, kiểm tra mỗi row để dừng sát ngưỡng hơn.
                should_check_size = (
                    PRIORITIZE_MB
                    and (
                        rows_written % 100 == 0
                        or file_size_mb >= LIMIT_MB * 0.95
                    )
                )

                if should_check_size:
                    f.flush()
                    file_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)
                    pbar.n = min(round(file_size_mb, 2), LIMIT_MB)
                    pbar.refresh()
                elif not PRIORITIZE_MB:
                    pbar.update(1)

                # Hard-stop theo MB nếu đang ưu tiên dung lượng.
                if PRIORITIZE_MB and file_size_mb >= LIMIT_MB:
                    print(
                        f"\n[DỪNG] Đã đạt giới hạn dung lượng: "
                        f"{file_size_mb:.2f} MB "
                        f"(Tổng: {rows_written:,} dòng)"
                    )
                    break

                # Hard-stop theo số dòng trong mọi chế độ.
                if rows_written >= LIMIT_ROWS:
                    f.flush()
                    file_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)
                    print(
                        f"\n[DỪNG] Đã đạt giới hạn số dòng: "
                        f"{rows_written:,} dòng "
                        f"(Dung lượng: {file_size_mb:.2f} MB)"
                    )
                    break

        f.flush()

    final_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)
    print(
        f"✅ Hoàn thành: {os.path.abspath(OUTPUT_CSV)}\n"
        f"   Rows: {rows_written:,}\n"
        f"   Size: {final_size_mb:.2f} MB"
    )

    # Đồng bộ đường dẫn cho cell loader tiếp theo.
    DATA_PATH = OUTPUT_CSV

except StopIteration:
    raise RuntimeError("Dataset stream rỗng: không lấy được dòng đầu tiên.")
except Exception as e:
    print(f"\n❌ Có lỗi xảy ra: {e}")
    print(
        "Kiểm tra: (1) HF_TOKEN, (2) quyền Agree/Access trên Hugging Face, "
        "(3) kết nối mạng của Colab."
    )
    raise

Đang kết nối luồng dữ liệu (streaming)...
Đang ghi dữ liệu vào: /content/hackernoon_subset.csv


Đang tải (MB):   0%|          | 0/150 [00:00<?, ?MB/s]


[DỪNG] Đã đạt giới hạn dung lượng: 150.00 MB (Tổng: 256,695 dòng)
✅ Hoàn thành: /content/hackernoon_subset.csv
   Rows: 256,695
   Size: 150.00 MB


In [26]:
#@title 1.4 — Neo4j connection + schema
driver = None

def connect_neo4j():
    global driver
    if not NEO4J_URI or not NEO4J_PASSWORD:
        raise ValueError("Thiếu Neo4j secrets.")
    driver = GraphDatabase.driver(
        NEO4J_URI,
        auth=(NEO4J_USER, NEO4J_PASSWORD),
    )
    driver.verify_connectivity()
    print("✅ Neo4j connected.")

def run_cypher(query, **params):
    if driver is None:
        raise RuntimeError("Hãy chạy connect_neo4j() trước.")
    with driver.session(database=NEO4J_DATABASE) as session:
        result = session.run(query, **params)
        rows = [r.data() for r in result]
        result.consume()
    return rows

def setup_graph_schema():
    for stmt in [
        """
        CREATE CONSTRAINT entity_id IF NOT EXISTS
        FOR (n:Entity) REQUIRE n.id IS UNIQUE
        """,
        """
        CREATE INDEX entity_name_norm IF NOT EXISTS
        FOR (n:Entity) ON (n.name_norm)
        """,
        """
        CREATE INDEX company_name_norm IF NOT EXISTS
        FOR (n:Company) ON (n.name_norm)
        """,
        """
        CREATE INDEX person_name_norm IF NOT EXISTS
        FOR (n:Person) ON (n.name_norm)
        """,
        """
        CREATE INDEX technology_name_norm IF NOT EXISTS
        FOR (n:Technology) ON (n.name_norm)
        """,
    ]:
        run_cypher(stmt)
    print("✅ Schema ready.")

connect_neo4j()
setup_graph_schema()


✅ Neo4j connected.
✅ Schema ready.


In [27]:
#@title 1.5 — Loader + exact dedup + chunking utilities
def norm_space(x):
    return re.sub(r"\s+", " ", str(x or "")).strip()

def sha1(x):
    return hashlib.sha1(str(x).encode("utf-8", errors="ignore")).hexdigest()

def pick_col(df, candidates, required=True):
    lookup = {str(c).lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lookup:
            return lookup[c.lower()]
    if required:
        raise KeyError(f"Missing one of columns: {candidates}. Available={list(df.columns)}")
    return None

def load_news(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() in {".jsonl", ".ndjson"}:
        return pd.read_json(path, lines=True)
    if path.suffix.lower() == ".json":
        return pd.read_json(path)
    if path.suffix.lower() in {".parquet", ".pq"}:
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported: {path.suffix}")

def standardize_news(raw):
    # HackerNoon's public dump commonly exposes article text as `description`.
    text_col = pick_col(raw, ["text", "content", "article", "body", "story", "description"])
    title_col = pick_col(raw, ["title", "headline"], required=False)
    date_col = pick_col(
        raw,
        ["published_date", "published_at", "published", "date", "created_at"],
        required=False
    )
    id_col = pick_col(raw, ["id", "_id", "article_id", "story_id", "uuid"], required=False)

    df = pd.DataFrame(index=raw.index)
    df["text"] = raw[text_col].fillna("").map(norm_space)
    df["title"] = raw[title_col].fillna("").map(norm_space) if title_col else ""

    if date_col:
        parsed = pd.to_datetime(raw[date_col], errors="coerce", utc=True)
        df["published_date"] = parsed.dt.strftime("%Y-%m-%d").fillna("unknown")
    else:
        df["published_date"] = "unknown"

    if id_col:
        # Handles scalar IDs as well as Mongo-style dict-like values.
        df["article_id"] = raw[id_col].map(lambda x: sha1(json.dumps(x, sort_keys=True, default=str))[:20])
    else:
        df["article_id"] = [
            sha1(f"{t}\n{x}")[:20] for t, x in zip(df["title"], df["text"])
        ]

    df = df[df["text"].str.len() >= 80].copy()
    df["dedup_key"] = [
        sha1(norm_space(f"{t}\n{x}").lower())
        for t, x in zip(df["title"], df["text"])
    ]
    before = len(df)
    df = df.drop_duplicates("dedup_key").drop(columns="dedup_key").reset_index(drop=True)
    print(f"Exact dedup: {before:,} -> {len(df):,}")

    if LAB_MAX_ARTICLES and len(df) > LAB_MAX_ARTICLES:
        df = (
            df.sample(LAB_MAX_ARTICLES, random_state=SEED)
              .sort_index()
              .reset_index(drop=True)
        )
    return df

def chunk_text(text, size=220, overlap=40):
    words = norm_space(text).split()
    step = max(1, size - overlap)
    out = []
    for start in range(0, len(words), step):
        part = words[start:start+size]
        if not part:
            break
        out.append(" ".join(part))
        if start + size >= len(words):
            break
    return out

def build_chunks(news_df):
    rows = []
    for r in tqdm(news_df.itertuples(index=False), total=len(news_df), desc="Chunking"):
        for i, text in enumerate(chunk_text(r.text, CHUNK_WORDS, CHUNK_OVERLAP_WORDS)):
            rows.append({
                "chunk_id": f"{r.article_id}::c{i:04d}",
                "article_id": r.article_id,
                "title": r.title,
                "published_date": r.published_date,
                "text": text,
            })
            if LAB_MAX_CHUNKS and len(rows) >= LAB_MAX_CHUNKS:
                return pd.DataFrame(rows)
    return pd.DataFrame(rows)

raw_df = load_news(DATA_PATH)
news_df = standardize_news(raw_df)
display(news_df.head())


Exact dedup: 121,179 -> 105,744


,text,title,published_date,article_id
0,The top email encryption services we''ve tested can help keep snoops out of your messages. When the IBM PC was new I...,The Best Email Encryption Services for 2023,2023-09-20,42ed41030306377a2734
1,After being held remotely due to COVID-19 in-person meetings are resuming as of Friday 4/20/23. All meetings will be...,Committee on Information Technology (COIT),2023-09-02,b6da26894a7757192ac5
2,(RTTNews) - Evernorth Health Services the pharmacy care and benefits solution division of The Cigna Group (CI) annou...,Evernorth Health Services Buys Bright.md Technology Platform,2023-10-10,2e7b477d4cf02b299c80
3,The Human Services Technology Addiction and Recovery Studies concentration prepares students to assist in drug and a...,Degrees & Pathways,2023-10-04,472078c40e5bc2233c3e
4,PARIS: With the exception of Canada countries with digital services taxes have agreed to hold off applying them for ...,Countries agree to extend digital services tax freeze through 2024,2023-07-17,56de0db4b3758fddcc82


### 🎯 AI Coding Agent Challenge A — Near Dedup
Exact hash không bắt được bài repost/near-duplicate.

Hãy dùng AI Agent thiết kế thêm **MinHash/LSH, SimHash hoặc embedding+ANN**.  
**Không chấp nhận** pairwise cosine `O(N²)` trên toàn dataset.

Trong báo cáo nêu:
1. threshold,
2. false positive,
3. cách audit cặp bị merge.

In [28]:
#@title 1.5B — Near-Dedup bằng MinHash/LSH (không O(N²))
from datasketch import MinHash, MinHashLSH

NEAR_DEDUP_LSH_THRESHOLD = 0.80
NEAR_DEDUP_VERIFY_JACCARD = 0.88
NEAR_DEDUP_NUM_PERM = 128
NEAR_DEDUP_SHINGLE_N = 5

def word_shingles(text, n=5):
    toks = re.findall(r"[a-z0-9]+", norm_space(text).lower())
    if len(toks) < n:
        return {" ".join(toks)} if toks else set()
    return {" ".join(toks[i:i+n]) for i in range(len(toks)-n+1)}

def minhash_from_shingles(shingles, num_perm=128):
    m = MinHash(num_perm=num_perm, seed=SEED)
    for s in shingles:
        m.update(s.encode("utf-8", errors="ignore"))
    return m

def exact_jaccard(a, b):
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)

class DedupUF:
    def __init__(self, n):
        self.p = list(range(n))
    def find(self, x):
        if self.p[x] != x:
            self.p[x] = self.find(self.p[x])
        return self.p[x]
    def union(self, a, b):
        a, b = self.find(a), self.find(b)
        if a != b:
            self.p[b] = a

def near_dedup_minhash(
    df,
    lsh_threshold=NEAR_DEDUP_LSH_THRESHOLD,
    verify_jaccard=NEAR_DEDUP_VERIFY_JACCARD,
    num_perm=NEAR_DEDUP_NUM_PERM,
    shingle_n=NEAR_DEDUP_SHINGLE_N,
):
    if df.empty:
        return df.copy(), pd.DataFrame()

    texts = [
        norm_space(f"{t} {x}")
        for t, x in zip(df["title"].fillna(""), df["text"].fillna(""))
    ]
    shingles = [word_shingles(x, shingle_n) for x in texts]
    minhashes = [minhash_from_shingles(s, num_perm) for s in tqdm(shingles, desc="MinHash")]

    lsh = MinHashLSH(threshold=lsh_threshold, num_perm=num_perm)
    uf = DedupUF(len(df))
    audit = []

    # Query only against already-indexed documents => candidate generation is subquadratic.
    for i, mh in enumerate(tqdm(minhashes, desc="LSH candidates")):
        for key in lsh.query(mh):
            j = int(key)
            jac = exact_jaccard(shingles[i], shingles[j])
            decision = "MERGE_NEAR_DUP" if jac >= verify_jaccard else "REJECT_VERIFY"
            audit.append({
                "left_row": j,
                "right_row": i,
                "left_article_id": df.iloc[j]["article_id"],
                "right_article_id": df.iloc[i]["article_id"],
                "jaccard": float(jac),
                "decision": decision,
            })
            if jac >= verify_jaccard:
                uf.union(i, j)
        lsh.insert(str(i), mh)

    groups = defaultdict(list)
    for i in range(len(df)):
        groups[uf.find(i)].append(i)

    keep = []
    for members in groups.values():
        # Keep the richest version; tie-break on earliest row for determinism.
        best = max(members, key=lambda i: (len(df.iloc[i]["text"]), -i))
        keep.append(best)

    out = df.iloc[sorted(keep)].reset_index(drop=True)
    audit_df = pd.DataFrame(audit)
    print(
        f"Near dedup: {len(df):,} -> {len(out):,} "
        f"(removed {len(df)-len(out):,}, verify_jaccard={verify_jaccard})"
    )
    if not audit_df.empty:
        print("Candidate decisions:", audit_df["decision"].value_counts().to_dict())
    return out, audit_df

news_df, near_dedup_audit_df = near_dedup_minhash(news_df)
chunks_df = build_chunks(news_df)

display(chunks_df.head())
if not near_dedup_audit_df.empty:
    display(near_dedup_audit_df.sort_values("jaccard", ascending=False).head(20))


MinHash:   0%|          | 0/1500 [00:00<?, ?it/s]

LSH candidates:   0%|          | 0/1500 [00:00<?, ?it/s]

Near dedup: 1,500 -> 1,500 (removed 0, verify_jaccard=0.88)
Candidate decisions: {'REJECT_VERIFY': 1}


Chunking:   0%|          | 0/1500 [00:00<?, ?it/s]

,chunk_id,article_id,title,published_date,text
0,42ed41030306377a2734::c0000,42ed41030306377a2734,The Best Email Encryption Services for 2023,2023-09-20,The top email encryption services we''ve tested can help keep snoops out of your messages. When the IBM PC was new I...
1,b6da26894a7757192ac5::c0000,b6da26894a7757192ac5,Committee on Information Technology (COIT),2023-09-02,After being held remotely due to COVID-19 in-person meetings are resuming as of Friday 4/20/23. All meetings will be...
2,2e7b477d4cf02b299c80::c0000,2e7b477d4cf02b299c80,Evernorth Health Services Buys Bright.md Technology Platform,2023-10-10,(RTTNews) - Evernorth Health Services the pharmacy care and benefits solution division of The Cigna Group (CI) annou...
3,472078c40e5bc2233c3e::c0000,472078c40e5bc2233c3e,Degrees & Pathways,2023-10-04,The Human Services Technology Addiction and Recovery Studies concentration prepares students to assist in drug and a...
4,56de0db4b3758fddcc82::c0000,56de0db4b3758fddcc82,Countries agree to extend digital services tax freeze through 2024,2023-07-17,PARIS: With the exception of Canada countries with digital services taxes have agreed to hold off applying them for ...


,left_row,right_row,left_article_id,right_article_id,jaccard,decision
0,1047,1048,6befc0e25a4b104ae7e2,298d51c63d0df9e33723,0.8,REJECT_VERIFY


In [29]:
#@title 1.6 — OpenRouter LLM wrapper có retry + JSON parsing
from openai import OpenAI

_openrouter_headers = {}
if OPENROUTER_HTTP_REFERER:
    _openrouter_headers["HTTP-Referer"] = OPENROUTER_HTTP_REFERER
if OPENROUTER_APP_TITLE:
    _openrouter_headers["X-OpenRouter-Title"] = OPENROUTER_APP_TITLE

openrouter_client = (
    OpenAI(
        api_key=OPENROUTER_API_KEY,
        base_url=OPENROUTER_BASE_URL,
        default_headers=_openrouter_headers or None,
    )
    if OPENROUTER_API_KEY
    else None
)

def parse_json_object(text):
    text = str(text).strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.I)
    text = re.sub(r"\s*```$", "", text)
    a, b = text.find("{"), text.rfind("}")
    if a < 0 or b <= a:
        raise ValueError("No JSON object found.")
    return json.loads(text[a:b+1])

def _usage_dict(resp):
    usage = getattr(resp, "usage", None)
    if usage is None:
        return {}
    return {
        "prompt_tokens": getattr(usage, "prompt_tokens", None),
        "completion_tokens": getattr(usage, "completion_tokens", None),
        "total_tokens": getattr(usage, "total_tokens", None),
    }

def openrouter_chat(messages, model=None, json_mode=False, max_retries=4):
    if openrouter_client is None:
        raise RuntimeError("Thiếu OPENROUTER_API_KEY.")

    model = model or OPENROUTER_MODEL
    if not model:
        raise RuntimeError("Thiếu OPENROUTER_MODEL. Hãy dùng model slug dạng provider/model-name.")

    last = None
    for attempt in range(max_retries):
        try:
            kwargs = {
                "model": model,
                "messages": messages,
                "temperature": 0.0,
            }
            if json_mode:
                kwargs["response_format"] = {"type": "json_object"}

            try:
                resp = openrouter_client.chat.completions.create(**kwargs)
            except Exception as first_error:
                # Một số model/provider trên OpenRouter không hỗ trợ response_format.
                # Prompt vẫn yêu cầu strict JSON, nên fallback sang plain chat completion.
                msg = str(first_error).lower()
                unsupported_json_mode = json_mode and any(
                    key in msg
                    for key in ["response_format", "json_object", "structured output", "unsupported parameter"]
                )
                if not unsupported_json_mode:
                    raise
                kwargs.pop("response_format", None)
                resp = openrouter_client.chat.completions.create(**kwargs)

            return resp.choices[0].message.content, _usage_dict(resp)

        except Exception as e:
            last = e
            if attempt == max_retries - 1:
                break
            time.sleep(min(20, 2**attempt + random.random()))

    raise RuntimeError(f"OpenRouter request failed after {max_retries} attempts: {last}")

def openrouter_json(system, user, model=None):
    text, usage = openrouter_chat(
        [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        model=model,
        json_mode=True,
    )
    return parse_json_object(text), usage


In [30]:
#@title 1.6B — OpenRouter configuration check (không gọi API)
assert OPENROUTER_BASE_URL == "https://openrouter.ai/api/v1"
print("OpenRouter endpoint:", OPENROUTER_BASE_URL)
print("API key configured:", bool(OPENROUTER_API_KEY))
print("Main model:", OPENROUTER_MODEL or "<set OPENROUTER_MODEL in Colab Secrets>")
print("Judge model:", OPENROUTER_JUDGE_MODEL or OPENROUTER_MODEL or "<same as main model>")


OpenRouter endpoint: https://openrouter.ai/api/v1
API key configured: True
Main model: poolside/laguna-xs-2.1
Judge model: poolside/laguna-xs-2.1


## 1.7 — Coreference Resolution

Yêu cầu:
- chỉ resolve đại từ khi antecedent rõ trong cùng chunk,
- không invent fact,
- giữ nguyên số/ngày/ticker/product,
- ambiguity → giữ nguyên và log `unresolved_mentions`.

**Failure mode quan trọng:** false coreference → false edge.

In [31]:
#@title 1.7 — Coreference resolution theo batch
COREF_SYSTEM = """
You are a conservative coreference-resolution component for a knowledge-graph pipeline.
Resolve pronouns and generic references only when the antecedent is clearly supported in the same chunk.
Never invent facts. Preserve dates, numbers, tickers and product names.
Return strict JSON only.
""".strip()

def resolve_coref_batch(batch_df):
    payload = [{"chunk_id": r.chunk_id, "text": r.text}
               for r in batch_df.itertuples(index=False)]

    prompt = f"""
Resolve coreferences.

Return:
{{
  "items": [
    {{
      "chunk_id": "...",
      "resolved_text": "...",
      "unresolved_mentions": ["..."]
    }}
  ]
}}

INPUT:
{json.dumps(payload, ensure_ascii=False)}
""".strip()

    obj, usage = openrouter_json(COREF_SYSTEM, prompt)
    by_id = {x.get("chunk_id"): x for x in obj.get("items", [])}

    rows = []
    for r in batch_df.itertuples(index=False):
        item = by_id.get(r.chunk_id, {})
        rows.append({
            "chunk_id": r.chunk_id,
            "resolved_text": norm_space(item.get("resolved_text") or r.text),
            "unresolved_mentions": item.get("unresolved_mentions", []),
        })
    return pd.DataFrame(rows), usage

def run_coref(chunks_subset, batch_size=5):
    out = []
    for start in tqdm(range(0, len(chunks_subset), batch_size), desc="Coref"):
        batch = chunks_subset.iloc[start:start+batch_size]
        try:
            df, _ = resolve_coref_batch(batch)
        except Exception:
            df = pd.DataFrame({
                "chunk_id": batch["chunk_id"].tolist(),
                "resolved_text": batch["text"].tolist(),
                "unresolved_mentions": [["COREF_BATCH_FAILED"] for _ in range(len(batch))],
            })
        out.append(df)
    return pd.concat(out, ignore_index=True)

extraction_source = chunks_df.head(EXTRACTION_MAX_CHUNKS).copy()
coref_df = run_coref(extraction_source)
extraction_source = extraction_source.merge(coref_df, on="chunk_id", how="left")

print("Coreference rows:", len(coref_df))
display(coref_df.head())


Coref:   0%|          | 0/80 [00:00<?, ?it/s]

Coreference rows: 400


,chunk_id,resolved_text,unresolved_mentions
0,42ed41030306377a2734::c0000,The top email encryption services we''ve tested can help keep snoops out of your messages. When the IBM PC was new I...,[]
1,b6da26894a7757192ac5::c0000,After being held remotely due to COVID-19 in-person meetings are resuming as of Friday 4/20/23. All meetings will be...,[]
2,2e7b477d4cf02b299c80::c0000,(RTTNews) - Evernorth Health Services the pharmacy care and benefits solution division of The Cigna Group (CI) annou...,[]
3,472078c40e5bc2233c3e::c0000,The Human Services Technology Addiction and Recovery Studies concentration prepares students to assist in drug and a...,[]
4,56de0db4b3758fddcc82::c0000,PARIS: With the exception of Canada countries with digital services taxes have agreed to hold off applying them for ...,[]


# PHẦN 2 — TRIPLE EXTRACTION & NEO4J BULK INSERT (15–45')

## Graph schema
**Nodes:** `Company`, `Person`, `Technology` + base label `Entity`.

**Relations:** `ACQUIRED`, `DEVELOPED`, `INVESTED_IN`, `FOUNDED`, `WORKED_AT`, `PARTNERED_WITH`, `USES`, `LEADS`.

**Mỗi edge bắt buộc:** `source_chunk_id`, `published_date`; khuyến nghị thêm `evidence`, `confidence`.

> Relation type phải qua allowlist trước khi ghép vào Cypher.

In [32]:
#@title 2.1 — NER + RE extraction
ALLOWED_NODE_TYPES = {"Company", "Person", "Technology"}
ALLOWED_RELATIONS = {
    "ACQUIRED", "DEVELOPED", "INVESTED_IN", "FOUNDED",
    "WORKED_AT", "PARTNERED_WITH", "USES", "LEADS"
}

EXTRACT_SYSTEM = f"""
Extract a high-precision knowledge graph from tech-news text.
Allowed node types: {sorted(ALLOWED_NODE_TYPES)}
Allowed relations: {sorted(ALLOWED_RELATIONS)}
Use only explicitly supported facts. Prefer precision over recall.
Every relation needs short evidence. Return strict JSON only.
""".strip()

def extract_batch(batch_df):
    payload = [{
        "chunk_id": r.chunk_id,
        "published_date": r.published_date,
        "text": getattr(r, "resolved_text", None) or r.text,
    } for r in batch_df.itertuples(index=False)]

    prompt = f"""
Return:
{{
  "items": [
    {{
      "chunk_id": "...",
      "relations": [
        {{
          "source": "...",
          "source_type": "Company|Person|Technology",
          "relation": "ALLOWED_RELATION",
          "target": "...",
          "target_type": "Company|Person|Technology",
          "evidence": "...",
          "confidence": 0.0
        }}
      ]
    }}
  ]
}}

INPUT:
{json.dumps(payload, ensure_ascii=False)}
""".strip()
    return openrouter_json(EXTRACT_SYSTEM, prompt)

def run_extraction(source_df, batch_size=4):
    meta = source_df.set_index("chunk_id")["published_date"].to_dict()
    triples, errors = [], []

    for start in tqdm(range(0, len(source_df), batch_size), desc="NER+RE"):
        batch = source_df.iloc[start:start+batch_size]
        try:
            obj, _ = extract_batch(batch)
        except Exception as e:
            errors.append({"start": start, "error": str(e)})
            continue

        for item in obj.get("items", []):
            cid = item.get("chunk_id")
            if cid not in meta:
                continue
            for x in item.get("relations", []):
                s, t = norm_space(x.get("source")), norm_space(x.get("target"))
                st, tt, rel = x.get("source_type"), x.get("target_type"), x.get("relation")
                if not s or not t:
                    continue
                if st not in ALLOWED_NODE_TYPES or tt not in ALLOWED_NODE_TYPES:
                    continue
                if rel not in ALLOWED_RELATIONS:
                    continue
                triples.append({
                    "source_raw": s,
                    "source_type": st,
                    "relation": rel,
                    "target_raw": t,
                    "target_type": tt,
                    "source_chunk_id": cid,
                    "published_date": meta[cid] or "unknown",
                    "evidence": norm_space(x.get("evidence")),
                    "confidence": max(0.0, min(1.0, float(x.get("confidence") or 0.0))),
                })

    return pd.DataFrame(triples), pd.DataFrame(errors)

raw_triples_df, extraction_errors_df = run_extraction(extraction_source)
print(f"Extracted triples: {len(raw_triples_df):,}; failed batches: {len(extraction_errors_df):,}")
display(raw_triples_df.head())
if not extraction_errors_df.empty:
    display(extraction_errors_df.head(20))


NER+RE:   0%|          | 0/100 [00:00<?, ?it/s]

Extracted triples: 157; failed batches: 3


,source_raw,source_type,relation,target_raw,target_type,source_chunk_id,published_date,evidence,confidence
0,Evernorth Health Services,Company,ACQUIRED,Bright.md,Company,2e7b477d4cf02b299c80::c0000,2023-10-10,Evernorth Health Services the pharmacy care and benefits solution division of The Cigna Group (CI) announced it will...,1.0
1,Williams,Person,WORKED_AT,EY Nigeria,Company,c1dd9a4010af3063c2a0::c0000,2023-07-17,EY Nigeria has announced the admission of four partners four associate partners and one director into its partnershi...,0.9
2,Syntech,Company,DEVELOPED,6-in-1 Multifunctional Docking Station,Technology,05830fc2165bfe224a68::c0000,2023-03-15,Syntech global digital accessories industry leader has announced the release of its latest recent product——the 6-in-...,0.9
3,Gatzby,Company,USES,AI,Technology,5dfa14cf784f2af7779f::c0000,2023-06-02,Gatzby's technology works with AI to forge social connections at the company's events by matching attendees on a per...,1.0
4,Mehrdad Radmehr,Person,LEADS,Teledyne Controls,Company,455c35de53cd094d519b::c0000,2023-06-08,'We are honored to be the recipient of such a prestigious industry award' said Mehrdad Radmehr President of Teledyne...,1.0


,start,error
0,104,No JSON object found.
1,220,No JSON object found.
2,236,No JSON object found.


## 2.2 — Entity Resolution bằng Vector Similarity

Pipeline:
1. Manual aliases cho ticker/tên rất phổ biến.
2. Embedding ANN candidate.
3. Lexical guard để giảm false merge.
4. Xuất audit table.

### 🎯 AI Coding Agent Challenge B
Cải tiến guard cho:
- ticker,
- suffix `Inc./Corp./Ltd.`,
- product chứa company name,
- người trùng họ/tên gần giống.

In [33]:
#@title 2.2 — Entity resolution
CORP_SUFFIXES = {"inc","incorporated","corp","corporation","ltd","limited","llc","plc","co","company"}
MANUAL_ALIASES = {
    ("Company", "msft"): "Microsoft",
    ("Company", "microsoft corp"): "Microsoft",
    ("Company", "microsoft corporation"): "Microsoft",
    ("Company", "goog"): "Google",
    ("Company", "googl"): "Google",
    ("Company", "google llc"): "Google",
    ("Company", "meta platforms"): "Meta",
    ("Company", "meta platforms inc"): "Meta",
    ("Company", "aapl"): "Apple",
    ("Company", "apple inc"): "Apple",
}

def norm_entity(name):
    s = unicodedata.normalize("NFKC", norm_space(name)).lower()
    s = re.sub(r"[^\w\s\-]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def strip_suffix(name):
    toks = norm_entity(name).split()
    while toks and toks[-1] in CORP_SUFFIXES:
        toks.pop()
    return " ".join(toks)

def person_core_tokens(name):
    toks = norm_entity(name).split()
    return toks

def merge_guard(a, b, typ):
    na, nb = norm_entity(a), norm_entity(b)
    if na == nb:
        return True, "EXACT_NORMALIZED"

    # Never cross entity types: candidates are built per-type.
    if typ == "Company":
        sa, sb = strip_suffix(a), strip_suffix(b)
        if sa == sb and sa:
            return True, "CORP_SUFFIX_ONLY"
        ratio = SequenceMatcher(None, sa, sb).ratio()
        # A company name contained inside a longer product-like phrase is suspicious.
        contained = (sa in sb or sb in sa) and min(len(sa), len(sb)) >= 4
        if contained and abs(len(sa.split()) - len(sb.split())) >= 2:
            return False, "PRODUCT_OR_DESCRIPTOR_GUARD"
        return ratio >= 0.82, f"COMPANY_LEXICAL_{ratio:.3f}"

    if typ == "Person":
        ta, tb = person_core_tokens(a), person_core_tokens(b)
        # Conservative: first and last name must agree; middle initials may differ.
        if len(ta) >= 2 and len(tb) >= 2 and ta[0] == tb[0] and ta[-1] == tb[-1]:
            return True, "PERSON_FIRST_LAST_MATCH"
        ratio = SequenceMatcher(None, na, nb).ratio()
        return False, f"PERSON_CONSERVATIVE_REJECT_{ratio:.3f}"

    if typ == "Technology":
        ratio = SequenceMatcher(None, na, nb).ratio()
        # Product/technology names are easy to over-merge; require very strong lexical agreement.
        return ratio >= 0.92, f"TECH_LEXICAL_{ratio:.3f}"

    return False, "UNKNOWN_TYPE"

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embedder = None

def get_embedder():
    global embedder
    if embedder is None:
        embedder = SentenceTransformer(EMBED_MODEL)
    return embedder

class UF:
    def __init__(self, n):
        self.p = list(range(n))
    def find(self, x):
        if self.p[x] != x:
            self.p[x] = self.find(self.p[x])
        return self.p[x]
    def union(self, a, b):
        a, b = self.find(a), self.find(b)
        if a != b:
            self.p[b] = a

def build_resolution_map(raw_triples_df, threshold=ENTITY_VECTOR_THRESHOLD, top_k=ENTITY_TOP_K):
    mentions = []
    for r in raw_triples_df.itertuples(index=False):
        mentions += [(r.source_type, r.source_raw), (r.target_type, r.target_raw)]

    counts = Counter((t, norm_entity(n)) for t, n in mentions)
    display_name = {}
    for t, n in mentions:
        display_name.setdefault((t, norm_entity(n)), n)

    mapping, audit = {}, []

    for key in counts:
        typ, norm = key
        manual = MANUAL_ALIASES.get((typ, norm))
        if manual:
            mapping[key] = manual
            audit.append({
                "type": typ, "left": display_name[key], "right": manual,
                "similarity": 1.0, "decision": "MERGE_MANUAL", "guard_reason": "MANUAL_ALIAS"
            })

    for typ in sorted(ALLOWED_NODE_TYPES):
        keys = [k for k in counts if k[0] == typ and k not in mapping]
        if not keys:
            continue

        names = [display_name[k] for k in keys]
        vecs = get_embedder().encode(
            names, batch_size=128, show_progress_bar=False,
            normalize_embeddings=True
        ).astype("float32")

        index = faiss.IndexFlatIP(vecs.shape[1])
        index.add(vecs)
        sims, nbrs = index.search(vecs, min(top_k, len(names)))
        uf = UF(len(names))

        for i in range(len(names)):
            for score, j in zip(sims[i], nbrs[i]):
                if j < 0 or i >= j or float(score) < threshold:
                    continue
                ok, reason = merge_guard(names[i], names[j], typ)
                audit.append({
                    "type": typ, "left": names[i], "right": names[j],
                    "similarity": float(score),
                    "decision": "MERGE_VECTOR" if ok else "REJECT_GUARD",
                    "guard_reason": reason,
                })
                if ok:
                    uf.union(i, j)

        groups = defaultdict(list)
        for i in range(len(names)):
            groups[uf.find(i)].append(i)

        for idxs in groups.values():
            best = sorted(
                idxs,
                key=lambda i: (-counts[keys[i]], len(names[i]), names[i].lower())
            )[0]
            canonical = names[best]
            for i in idxs:
                mapping[keys[i]] = canonical

    for key in counts:
        mapping.setdefault(key, display_name[key])

    return mapping, pd.DataFrame(audit)

def canonicalize_triples(raw_df, mapping):
    df = raw_df.copy()

    def canon(name, typ):
        n = norm_entity(name)
        return mapping.get((typ, n), MANUAL_ALIASES.get((typ, n), name))

    df["source_name"] = [canon(n,t) for n,t in zip(df.source_raw, df.source_type)]
    df["target_name"] = [canon(n,t) for n,t in zip(df.target_raw, df.target_type)]
    df["source_name_norm"] = df.source_name.map(norm_entity)
    df["target_name_norm"] = df.target_name.map(norm_entity)
    df["source_id"] = [sha1(f"{t}:{n}")[:24] for t,n in zip(df.source_type, df.source_name_norm)]
    df["target_id"] = [sha1(f"{t}:{n}")[:24] for t,n in zip(df.target_type, df.target_name_norm)]
    return df[df.source_id != df.target_id].reset_index(drop=True)

entity_map, entity_resolution_audit_df = build_resolution_map(raw_triples_df)
triples_df = canonicalize_triples(raw_triples_df, entity_map)

print(f"Canonical triples: {len(triples_df):,}; audit pairs: {len(entity_resolution_audit_df):,}")
display(entity_resolution_audit_df.head(20))


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Canonical triples: 157; audit pairs: 0


""


In [34]:
#@title 2.3 — Node table + UNWIND bulk insert
def build_nodes(triples_df):
    rows = []
    for r in triples_df.itertuples(index=False):
        rows += [
            {"id":r.source_id,"name":r.source_name,"name_norm":r.source_name_norm,"type":r.source_type,"alias":r.source_raw},
            {"id":r.target_id,"name":r.target_name,"name_norm":r.target_name_norm,"type":r.target_type,"alias":r.target_raw},
        ]
    tmp = pd.DataFrame(rows)
    if tmp.empty:
        return tmp

    out = []
    for (node_id,name,name_norm,typ), g in tmp.groupby(["id","name","name_norm","type"]):
        aliases = sorted(set(g["alias"].map(norm_space)))
        out.append({
            "id":node_id, "name":name, "name_norm":name_norm, "type":typ,
            "aliases":aliases,
            "aliases_norm":sorted(set(norm_entity(x) for x in aliases))
        })
    return pd.DataFrame(out)

def batches(records, size=1000):
    for i in range(0, len(records), size):
        yield records[i:i+size]

def bulk_insert_nodes(nodes_df, batch_size=1000):
    for typ in sorted(ALLOWED_NODE_TYPES):
        part = nodes_df[nodes_df.type == typ]
        if part.empty:
            continue
        query = f"""
        UNWIND $rows AS row
        MERGE (n:Entity {{id: row.id}})
        SET n:{typ},
            n.name=row.name,
            n.name_norm=row.name_norm,
            n.entity_type=row.type,
            n.aliases=row.aliases,
            n.aliases_norm=row.aliases_norm
        """
        for b in batches(part.to_dict("records"), batch_size):
            run_cypher(query, rows=b)

def bulk_insert_edges(triples_df, batch_size=1000):
    required = {"source_chunk_id","published_date"}
    if not required.issubset(triples_df.columns):
        raise ValueError("Missing edge provenance.")

    for rel in sorted(ALLOWED_RELATIONS):
        part = triples_df[triples_df.relation == rel]
        if part.empty:
            continue

        query = f"""
        UNWIND $rows AS row
        MATCH (s:Entity {{id: row.source_id}})
        MATCH (t:Entity {{id: row.target_id}})
        MERGE (s)-[r:{rel} {{source_chunk_id: row.source_chunk_id}}]->(t)
        SET r.published_date=row.published_date,
            r.evidence=row.evidence,
            r.confidence=row.confidence
        """

        cols = ["source_id","target_id","source_chunk_id","published_date","evidence","confidence"]
        for b in batches(part[cols].to_dict("records"), batch_size):
            run_cypher(query, rows=b)

nodes_df = build_nodes(triples_df)
if nodes_df.empty:
    raise ValueError("No nodes were produced. Check extraction results before inserting into Neo4j.")
bulk_insert_nodes(nodes_df)
bulk_insert_edges(triples_df)
print(f"Inserted/merged {len(nodes_df):,} nodes and {len(triples_df):,} provenance edges.")


Inserted/merged 251 nodes and 157 provenance edges.


In [35]:
#@title 2.4 — Sanity checks
def graph_checks():
    invalid = run_cypher("""
    MATCH ()-[r]->()
    WHERE r.source_chunk_id IS NULL OR trim(toString(r.source_chunk_id)) = ''
       OR r.published_date IS NULL OR trim(toString(r.published_date)) = ''
    RETURN count(r) AS n
    """)[0]["n"]

    counts = {
        "nodes": run_cypher("MATCH (n:Entity) RETURN count(n) AS n")[0]["n"],
        "edges": run_cypher("MATCH ()-[r]->() RETURN count(r) AS n")[0]["n"],
        "invalid_provenance_edges": invalid,
    }
    print(counts)
    assert invalid == 0

    top = pd.DataFrame(run_cypher("""
    MATCH (n:Entity)
    OPTIONAL MATCH (n)-[r]-()
    WITH n, count(r) AS degree
    RETURN n.id AS id, n.name AS name, n.entity_type AS type, degree
    ORDER BY degree DESC LIMIT 15
    """))
    display(top)
    return counts, top

graph_counts, top_degree_df = graph_checks()


{'nodes': 276, 'edges': 180, 'invalid_provenance_edges': 0}


,id,name,type,degree
0,e6c3db5c28c9a15bfafd2e04,Apple,Company,10
1,fe9221520afe23f76d432625,Databricks Lakehouse for Manufacturing,Technology,10
2,fb0f4df56fab164ec48722f0,Microsoft,Company,7
3,b0f073fc691ac6ea5bfa2c41,Activision Blizzard,Company,6
4,ed73b9eff3f5ff1eea6154b7,Xi Jinping,Person,5
5,037c23982154ce4d78f70913,Inflection AI,Company,4
6,4841986fd3829535095242bc,Parks Associates,Company,4
7,1029cc5a61d01976b9615863,1Kosmos,Company,4
8,11a9131cc2990b46b25c887b,LogistiVIEW,Company,3
9,160b234e1832ea41fbd6f3e2,Accessory,Technology,3


# PHẦN 3 — FLAT RAG & HYBRID GRAPHRAG (45–75')

## Flat RAG baseline
Dùng cùng embedding/generator để comparison tập trung vào retrieval architecture.

In [36]:
#@title 3.1 — Flat RAG
flat_index = None
flat_store = None
entity_match_vectors = None
entity_match_store = None

def build_flat_index(chunks_df):
    global flat_index, flat_store
    vecs = get_embedder().encode(
        chunks_df.text.fillna("").tolist(),
        batch_size=128, show_progress_bar=True,
        normalize_embeddings=True
    ).astype("float32")

    flat_index = faiss.IndexFlatIP(vecs.shape[1])
    flat_index.add(vecs)
    flat_store = chunks_df.reset_index(drop=True).copy()
    print("Flat vectors:", flat_index.ntotal)

def retrieve_flat_context(query, k=6):
    if flat_index is None or flat_store is None:
        raise RuntimeError("Flat index is not built. Run build_flat_index(chunks_df) first.")
    qv = get_embedder().encode(
        [query], normalize_embeddings=True, show_progress_bar=False
    ).astype("float32")
    scores, ids = flat_index.search(qv, min(k, flat_index.ntotal))

    rows = []
    for score, idx in zip(scores[0], ids[0]):
        if idx < 0:
            continue
        r = flat_store.iloc[int(idx)]
        rows.append({
            "score":float(score), "chunk_id":r.chunk_id,
            "published_date":r.published_date, "text":r.text
        })

    df = pd.DataFrame(rows)
    context = "\n\n".join(
        f"[chunk_id={r.chunk_id} | date={r.published_date} | score={r.score:.3f}]\n{r.text}"
        for r in df.itertuples(index=False)
    )
    return context, df

build_flat_index(chunks_df)


Batches:   0%|          | 0/12 [00:00<?, ?it/s]

Flat vectors: 1500


## Graph retrieval flow
1. LLM trích seed entities.
2. Match seed trong Neo4j; fuzzy fallback bằng embedding.
3. BFS tối đa `max_hops`.
4. Nếu node degree > 100 → chỉ lấy tối đa 50 edge mới nhất.
5. Global edge cap để tránh context explosion.
6. Textualize subgraph có provenance.

In [37]:
#@title 3.2 — Seed matching
SEED_SYSTEM = """
Extract useful seed entities for graph retrieval.
Allowed types: Company, Person, Technology.
Do not answer the question. Return strict JSON only.
""".strip()

def extract_seeds(query):
    obj, _ = openrouter_json(SEED_SYSTEM, f"""
Question: {query}
Return {{"seeds":[{{"name":"...","type":"Company|Person|Technology|null"}}]}}
""")
    return [
        {"name":norm_space(x.get("name")),
         "type":x.get("type") if x.get("type") in ALLOWED_NODE_TYPES else None}
        for x in obj.get("seeds", [])
        if norm_space(x.get("name"))
    ]

def build_entity_matcher(nodes_df):
    global entity_match_vectors, entity_match_store
    entity_match_store = nodes_df.reset_index(drop=True).copy()
    entity_match_vectors = get_embedder().encode(
        entity_match_store.name.tolist(),
        batch_size=128, show_progress_bar=False,
        normalize_embeddings=True
    ).astype("float32")

def match_seeds(query, fuzzy_threshold=SEED_FUZZY_THRESHOLD):
    matched = []
    for seed in extract_seeds(query):
        exact = run_cypher("""
        MATCH (n:Entity)
        WHERE (n.name_norm=$name OR $name IN coalesce(n.aliases_norm,[]))
          AND ($typ IS NULL OR n.entity_type=$typ)
        RETURN n.id AS id, n.name AS name, n.entity_type AS type
        LIMIT 5
        """, name=norm_entity(seed["name"]), typ=seed["type"])

        if exact:
            matched += exact
            continue

        if entity_match_vectors is None:
            continue

        mask = np.ones(len(entity_match_store), dtype=bool)
        if seed["type"]:
            mask = entity_match_store.type.eq(seed["type"]).to_numpy()
        idxs = np.flatnonzero(mask)
        if not len(idxs):
            continue

        qv = get_embedder().encode(
            [seed["name"]], normalize_embeddings=True, show_progress_bar=False
        ).astype("float32")[0]
        sims = entity_match_vectors[idxs] @ qv
        j = int(np.argmax(sims))
        if float(sims[j]) >= fuzzy_threshold:
            r = entity_match_store.iloc[int(idxs[j])]
            matched.append({"id":r.id,"name":r.name,"type":r.type})

    return list({x["id"]: x for x in matched}.values())

build_entity_matcher(nodes_df)


In [38]:
#@title 3.3 — Graph traversal + super-node mitigation
SUPER_NODE_DEGREE = 100
SUPER_NODE_EDGE_CAP = 50
GLOBAL_EDGE_CAP = 250
MAX_GRAPH_CONTEXT_CHARS = 14000

def node_degree(node_id):
    return int(run_cypher("""
    MATCH (n:Entity {id:$id})
    OPTIONAL MATCH (n)-[r]-()
    RETURN count(r) AS degree
    """, id=node_id)[0]["degree"])

def recent_edges(node_id, limit):
    return run_cypher("""
    MATCH (n:Entity {id:$id})
    MATCH (n)-[r]-(m:Entity)
    RETURN
      startNode(r).id AS source_id,
      startNode(r).name AS source_name,
      startNode(r).entity_type AS source_type,
      type(r) AS relation,
      endNode(r).id AS target_id,
      endNode(r).name AS target_name,
      endNode(r).entity_type AS target_type,
      r.source_chunk_id AS source_chunk_id,
      r.published_date AS published_date,
      r.evidence AS evidence,
      m.id AS neighbor_id
    ORDER BY CASE WHEN coalesce(r.published_date,'') = 'unknown' THEN '' ELSE coalesce(r.published_date,'') END DESC
    LIMIT $limit
    """, id=node_id, limit=int(limit))

def textualize(edges):
    edges = sorted(edges, key=lambda e:e.get("published_date") or "", reverse=True)
    lines, used = [], 0
    for e in edges:
        line = (
            f"{e['source_name']} [{e['source_type']}] -{e['relation']}-> "
            f"{e['target_name']} [{e['target_type']}] "
            f"| date={e.get('published_date') or 'unknown'} "
            f"| chunk={e.get('source_chunk_id') or 'unknown'}"
        )
        if e.get("evidence"):
            line += f" | evidence={norm_space(e['evidence'])}"
        if used + len(line) + 1 > MAX_GRAPH_CONTEXT_CHARS:
            break
        lines.append(line)
        used += len(line) + 1
    return "\n".join(lines)

def retrieve_graph_context(query, max_hops=2, edge_limit=100, return_debug=False):
    seeds = match_seeds(query)
    if not seeds:
        out = {"context":"","edges":pd.DataFrame(),
               "diagnostics":{"reason":"NO_SEED","supernode_events":[]}}
        return out if return_debug else ""

    frontier = deque((x["id"],0) for x in seeds)
    expanded, seen_edges, collected = set(), set(), []
    supernode_events = []

    while frontier and len(collected) < GLOBAL_EDGE_CAP:
        node_id, hop = frontier.popleft()
        if node_id in expanded or hop >= max_hops:
            continue
        expanded.add(node_id)

        degree = node_degree(node_id)
        limit = int(edge_limit)
        if degree > SUPER_NODE_DEGREE:
            limit = min(limit, SUPER_NODE_EDGE_CAP)
            supernode_events.append({"node_id":node_id,"degree":degree,"limit":limit})

        for e in recent_edges(node_id, limit):
            key = (e["source_id"],e["relation"],e["target_id"],e["source_chunk_id"])
            if key in seen_edges:
                continue
            seen_edges.add(key)
            collected.append(e)
            if len(collected) >= GLOBAL_EDGE_CAP:
                break

            nb = e.get("neighbor_id")
            if nb and nb not in expanded and hop + 1 < max_hops:
                frontier.append((nb, hop+1))

    out = {
        "context": textualize(collected),
        "edges": pd.DataFrame(collected),
        "diagnostics": {
            "matched_seeds": seeds,
            "expanded_nodes": len(expanded),
            "collected_edges": len(collected),
            "supernode_events": supernode_events,
        }
    }
    return out if return_debug else out["context"]


In [39]:
#@title 3.4 — Flat answer vs Hybrid GraphRAG answer
ANSWER_SYSTEM = """
Answer only from supplied context.
Be concise but complete. Do not invent facts.
Cite provenance inline as [chunk_id=...] whenever possible.
If evidence is insufficient or conflicting, say so.
""".strip()

def generate_answer(question, context):
    prompt = f"QUESTION:\n{question}\n\nCONTEXT:\n{context}\n\nANSWER:"
    t0 = time.perf_counter()
    text, usage = openrouter_chat(
        [{"role":"system","content":ANSWER_SYSTEM},
         {"role":"user","content":prompt}],
        model=OPENROUTER_MODEL
    )
    return {
        "answer": text.strip(),
        "latency_s": time.perf_counter()-t0,
        "total_tokens": usage.get("total_tokens"),
    }

def answer_flat_rag(question):
    context, retrieved = retrieve_flat_context(question, k=6)
    out = generate_answer(question, context)
    out.update({"context":context,"retrieved":retrieved})
    return out

def answer_graph_rag(question):
    g = retrieve_graph_context(question, max_hops=2, edge_limit=100, return_debug=True)
    vctx, vdocs = retrieve_flat_context(question, k=4)
    context = f"=== GRAPH ===\n{g['context']}\n\n=== VECTOR ===\n{vctx}"
    out = generate_answer(question, context)
    out.update({"context":context,"graph_debug":g,"vector_docs":vdocs})
    return out


# PHẦN 4 — GOLDEN DATASET & LLM-AS-A-JUDGE (75–105')

## Golden schema
`id`, `group`, `question`, `reference_answer`, optional `reference_evidence`.

Notebook có 5 câu starter. Các câu phụ thuộc data dump phải điền gold answer thật trước final evaluation.

In [40]:
#@title 4.1 — Golden Dataset: load verified CSV hoặc bootstrap 5 câu từ evidence hiện có
GOLDEN_PATH = "/content/golden_dataset.csv"

RELATION_TEXT = {
    "ACQUIRED": "acquired",
    "DEVELOPED": "developed",
    "INVESTED_IN": "invested in",
    "FOUNDED": "founded",
    "WORKED_AT": "worked at",
    "PARTNERED_WITH": "partnered with",
    "USES": "uses",
    "LEADS": "leads",
}

def edge_fact_text(r):
    rel = RELATION_TEXT.get(r.relation, r.relation.lower())
    return (
        f"{r.source_name} {rel} {r.target_name} "
        f"(date={r.published_date}, chunk={r.source_chunk_id})"
    )

def build_auto_golden(triples_df, n=5):
    """Bootstrap data-dependent questions from extracted evidence.

    This makes the notebook runnable on arbitrary subsets. For a real benchmark,
    manually verify these rows against the raw source chunks and set gold_verified=True,
    or provide /content/golden_dataset.csv curated independently.
    """
    if triples_df.empty:
        raise ValueError("Cannot build Golden Dataset: triples_df is empty.")

    e = triples_df.copy()
    e["confidence"] = pd.to_numeric(e["confidence"], errors="coerce").fillna(0.0)
    e = e.sort_values(["confidence", "published_date"], ascending=[False, False]).reset_index(drop=True)
    rows = []

    # G01 — factoid from a high-confidence edge.
    r = e.iloc[0]
    rel = RELATION_TEXT.get(r.relation, r.relation.lower())
    rows.append({
        "id": "G01",
        "group": "factoid",
        "question": f"According to the corpus, which entity is the target of {r.source_name} -{r.relation}-> ?",
        "reference_answer": r.target_name,
        "reference_evidence": edge_fact_text(r),
        "gold_verified": False,
        "gold_origin": "AUTO_FROM_EXTRACTED_EVIDENCE",
    })

    # G02 — directed two-hop chain A -> B -> C, preferably across documents.
    hop = e.merge(
        e,
        left_on="target_id",
        right_on="source_id",
        suffixes=("_1", "_2")
    )
    hop = hop[
        (hop["source_id_1"] != hop["target_id_2"]) &
        (hop["source_chunk_id_1"] != hop["source_chunk_id_2"])
    ]
    if not hop.empty:
        h = hop.sort_values(["confidence_1", "confidence_2"], ascending=False).iloc[0]
        rows.append({
            "id": "G02",
            "group": "multi-hop",
            "question": (
                f"Starting from {h.source_name_1}, follow two explicit graph relations. "
                f"Which intermediate entity and final entity are reached?"
            ),
            "reference_answer": (
                f"{h.source_name_1} -{h.relation_1}-> {h.target_name_1}; "
                f"{h.source_name_2} -{h.relation_2}-> {h.target_name_2}."
            ),
            "reference_evidence": (
                f"{h.evidence_1} [chunk_id={h.source_chunk_id_1}, date={h.published_date_1}] | "
                f"{h.evidence_2} [chunk_id={h.source_chunk_id_2}, date={h.published_date_2}]"
            ),
            "gold_verified": False,
            "gold_origin": "AUTO_FROM_EXTRACTED_EVIDENCE",
        })

    # G03 — two distinct chunks connected to the same source entity.
    multi = (
        e.groupby(["source_id", "source_name"])
         .filter(lambda g: g["source_chunk_id"].nunique() >= 2)
    )
    if not multi.empty:
        counts = (
            multi.groupby(["source_id", "source_name"])["source_chunk_id"]
                 .nunique()
                 .sort_values(ascending=False)
        )
        sid, sname = counts.index[0]
        g = (
            multi[multi["source_id"] == sid]
            .drop_duplicates("source_chunk_id")
            .sort_values("published_date")
            .head(2)
        )
        a, b = list(g.itertuples(index=False))
        rows.append({
            "id": "G03",
            "group": "cross-doc",
            "question": (
                f"Using evidence from at least two chunks, summarize two documented relations "
                f"involving {sname} and place them in chronological order when dates are available."
            ),
            "reference_answer": f"1) {edge_fact_text(a)}; 2) {edge_fact_text(b)}.",
            "reference_evidence": (
                f"{a.evidence} [chunk_id={a.source_chunk_id}] | "
                f"{b.evidence} [chunk_id={b.source_chunk_id}]"
            ),
            "gold_verified": False,
            "gold_origin": "AUTO_FROM_EXTRACTED_EVIDENCE",
        })

    # G04 — converging evidence: A -> X and B -> X.
    conv = e.merge(
        e,
        on="target_id",
        suffixes=("_1", "_2")
    )
    conv = conv[
        (conv["source_id_1"] < conv["source_id_2"]) &
        (conv["source_chunk_id_1"] != conv["source_chunk_id_2"])
    ]
    if not conv.empty:
        c = conv.sort_values(["confidence_1", "confidence_2"], ascending=False).iloc[0]
        rows.append({
            "id": "G04",
            "group": "multi-hop",
            "question": (
                f"Which shared entity is connected to both {c.source_name_1} and {c.source_name_2}, "
                f"and what are the two relation types?"
            ),
            "reference_answer": (
                f"The shared entity is {c.target_name_1}. "
                f"{c.source_name_1} -{c.relation_1}-> {c.target_name_1}; "
                f"{c.source_name_2} -{c.relation_2}-> {c.target_name_2}."
            ),
            "reference_evidence": (
                f"{c.evidence_1} [chunk_id={c.source_chunk_id_1}, date={c.published_date_1}] | "
                f"{c.evidence_2} [chunk_id={c.source_chunk_id_2}, date={c.published_date_2}]"
            ),
            "gold_verified": False,
            "gold_origin": "AUTO_FROM_EXTRACTED_EVIDENCE",
        })

    # G05 — repeated pair/relation across multiple chunks; otherwise another cross-doc entity.
    repeated = (
        e.groupby(["source_id", "relation", "target_id"])
         .filter(lambda g: g["source_chunk_id"].nunique() >= 2)
    )
    if not repeated.empty:
        key_counts = (
            repeated.groupby(["source_id", "relation", "target_id"])["source_chunk_id"]
                    .nunique()
                    .sort_values(ascending=False)
        )
        key = key_counts.index[0]
        g = (
            repeated[
                (repeated["source_id"] == key[0]) &
                (repeated["relation"] == key[1]) &
                (repeated["target_id"] == key[2])
            ]
            .drop_duplicates("source_chunk_id")
            .sort_values("published_date")
            .head(2)
        )
        a, b = list(g.itertuples(index=False))
        rows.append({
            "id": "G05",
            "group": "cross-doc",
            "question": (
                f"The corpus mentions the relation {a.source_name} -{a.relation}-> {a.target_name} "
                f"in multiple chunks. What evidence appears in two separate chunks, and how do their dates compare?"
            ),
            "reference_answer": (
                f"{edge_fact_text(a)}; {edge_fact_text(b)}."
            ),
            "reference_evidence": (
                f"{a.evidence} [chunk_id={a.source_chunk_id}] | "
                f"{b.evidence} [chunk_id={b.source_chunk_id}]"
            ),
            "gold_verified": False,
            "gold_origin": "AUTO_FROM_EXTRACTED_EVIDENCE",
        })

    # Ensure exactly n rows even on sparse graphs.
    used_signatures = {
        (x["question"], x["reference_answer"]) for x in rows
    }
    for r in e.itertuples(index=False):
        if len(rows) >= n:
            break
        rel = RELATION_TEXT.get(r.relation, r.relation.lower())
        candidate = {
            "id": f"G{len(rows)+1:02d}",
            "group": "factoid-fallback",
            "question": f"According to the cited chunk, which entity is the target of {r.source_name} -{r.relation}-> ?",
            "reference_answer": r.target_name,
            "reference_evidence": edge_fact_text(r),
            "gold_verified": False,
            "gold_origin": "AUTO_FROM_EXTRACTED_EVIDENCE",
        }
        sig = (candidate["question"], candidate["reference_answer"])
        if sig not in used_signatures:
            rows.append(candidate)
            used_signatures.add(sig)

    out = pd.DataFrame(rows[:n])
    out["id"] = [f"G{i+1:02d}" for i in range(len(out))]
    return out

if Path(GOLDEN_PATH).exists():
    golden_df = pd.read_csv(GOLDEN_PATH)
    print(f"Loaded curated Golden Dataset from {GOLDEN_PATH}")
else:
    golden_df = build_auto_golden(triples_df, n=5)
    golden_df.to_csv("/content/golden_dataset_bootstrap.csv", index=False)
    print(
        "⚠️ No curated golden_dataset.csv found. "
        "Created /content/golden_dataset_bootstrap.csv from extracted evidence. "
        "Review source chunks and set gold_verified=True before treating scores as final benchmark results."
    )

display(golden_df)

def validate_golden(df, require_answers=True, require_verified=False):
    required = {"id","group","question","reference_answer","reference_evidence"}
    if not required.issubset(df.columns):
        raise ValueError(f"Missing columns: {required-set(df.columns)}")
    if require_answers and df["reference_answer"].fillna("").str.strip().eq("").any():
        raise ValueError("Golden Dataset contains blank reference_answer values.")
    if df["reference_evidence"].fillna("").str.strip().eq("").any():
        raise ValueError("Golden Dataset contains blank reference_evidence values.")

    if require_verified:
        if "gold_verified" not in df.columns or not df["gold_verified"].fillna(False).astype(bool).all():
            raise ValueError(
                "Final benchmark requires manually verified gold rows. "
                "Review evidence and set gold_verified=True (or load a curated CSV)."
            )
    elif "gold_verified" in df.columns and not df["gold_verified"].fillna(False).astype(bool).all():
        print("⚠️ Golden rows are structurally complete but not all manually verified.")

    print("✅ Golden Dataset valid.")

validate_golden(golden_df, require_answers=True, require_verified=False)


⚠️ No curated golden_dataset.csv found. Created /content/golden_dataset_bootstrap.csv from extracted evidence. Review source chunks and set gold_verified=True before treating scores as final benchmark results.


,id,group,question,reference_answer,reference_evidence,gold_verified,gold_origin
0,G01,factoid,"According to the corpus, which entity is the target of Noah Kerner -LEADS-> ?",Acorns,"Noah Kerner leads Acorns (date=2023-10-19, chunk=0d2f4832beab70ebd1e9::c0000)",False,AUTO_FROM_EXTRACTED_EVIDENCE
1,G02,multi-hop,"Starting from Sheetal Elangovan, follow two explicit graph relations. Which intermediate entity and final entity are...",Sheetal Elangovan -WORKED_AT-> 1Kosmos; 1Kosmos -DEVELOPED-> BlockID Platform.,"Product Manager for 1Kosmos [chunk_id=ca988ab900607783b996::c0000, date=2023-09-06] | Company’s BlockID Platform cre...",False,AUTO_FROM_EXTRACTED_EVIDENCE
2,G03,cross-doc,"Using evidence from at least two chunks, summarize two documented relations involving Microsoft and place them in ch...","1) Microsoft acquired Activision Blizzard (date=2023-01-27, chunk=344312edbb3c92e1e2c8::c0000); 2) Microsoft acquire...",Microsoft’s proposed buyout of publishing giant Activision Blizzard [chunk_id=344312edbb3c92e1e2c8::c0000] | Microso...,False,AUTO_FROM_EXTRACTED_EVIDENCE
3,G04,multi-hop,"Which shared entity is connected to both Gatzby and Databricks Lakehouse for Manufacturing, and what are the two rel...",The shared entity is AI. Gatzby -USES-> AI; Databricks Lakehouse for Manufacturing -USES-> AI.,Gatzby's technology works with AI to forge social connections at the company's events by matching attendees on a per...,False,AUTO_FROM_EXTRACTED_EVIDENCE
4,G05,cross-doc,The corpus mentions the relation Microsoft -ACQUIRED-> Activision Blizzard in multiple chunks. What evidence appears...,"Microsoft acquired Activision Blizzard (date=2023-01-27, chunk=344312edbb3c92e1e2c8::c0000); Microsoft acquired Acti...",Microsoft’s proposed buyout of publishing giant Activision Blizzard [chunk_id=344312edbb3c92e1e2c8::c0000] | Microso...,False,AUTO_FROM_EXTRACTED_EVIDENCE


⚠️ Golden rows are structurally complete but not all manually verified.
✅ Golden Dataset valid.


In [41]:
#@title 4.2 — LLM-as-a-Judge via OpenRouter
JUDGE_SYSTEM = """
You are a strict evaluator of RAG answers.
Score 1-5:
- comprehensiveness
- faithfulness to supplied candidate context
- multi_hop_reasoning accuracy
Use the reference answer as correctness anchor.
Return strict JSON only.
""".strip()

def judge_json(system, user):
    model = OPENROUTER_JUDGE_MODEL or OPENROUTER_MODEL
    if not model:
        raise RuntimeError(
            "Thiếu OPENROUTER_JUDGE_MODEL/OPENROUTER_MODEL cho LLM-as-a-Judge."
        )
    return openrouter_json(system, user, model=model)[0]

def judge_answer(question, reference, answer, context):
    prompt = f"""
QUESTION:
{question}

REFERENCE:
{reference}

CANDIDATE:
{answer}

CANDIDATE CONTEXT:
{context[:18000]}

Return:
{{
 "comprehensiveness":1,
 "faithfulness":1,
 "multi_hop_reasoning":1,
 "rationale":"2-5 sentences"
}}
"""
    obj = judge_json(JUDGE_SYSTEM, prompt)
    out = {}
    for k in ["comprehensiveness","faithfulness","multi_hop_reasoning"]:
        out[k] = max(1, min(5, int(obj.get(k,1))))
    out["rationale"] = norm_space(obj.get("rationale"))
    return out


In [42]:
#@title 4.3 — Evaluation runner + checkpoint/resume
CHECKPOINT = "/content/graphrag_eval_checkpoint.csv"

def run_evaluation(golden_df, resume=True):
    existing = pd.DataFrame()
    if resume and Path(CHECKPOINT).exists():
        try:
            existing = pd.read_csv(CHECKPOINT)
            # Reuse only rows whose ID and question still match the current Golden Dataset.
            valid_pairs = set(zip(golden_df["id"].astype(str), golden_df["question"].astype(str)))
            existing = existing[
                [
                    (str(i), str(q)) in valid_pairs
                    for i, q in zip(existing["id"], existing["question"])
                ]
            ].copy()
            if not existing.empty:
                print(f"Resuming from {len(existing)} valid checkpoint rows.")
        except Exception as e:
            print("Checkpoint ignored:", e)
            existing = pd.DataFrame()

    done_ids = set(existing["id"].astype(str)) if not existing.empty else set()
    rows = existing.to_dict("records") if not existing.empty else []

    pending = golden_df[~golden_df["id"].astype(str).isin(done_ids)]
    for q in tqdm(pending.itertuples(index=False), total=len(pending), desc="Evaluation"):
        flat = answer_flat_rag(q.question)
        graph = answer_graph_rag(q.question)

        jf = judge_answer(q.question, q.reference_answer, flat["answer"], flat["context"])
        jg = judge_answer(q.question, q.reference_answer, graph["answer"], graph["context"])

        rows.append({
            "id":q.id, "group":q.group, "question":q.question,
            "reference_answer":q.reference_answer,
            "flat_answer":flat["answer"], "graph_answer":graph["answer"],
            "flat_comprehensiveness":jf["comprehensiveness"],
            "graph_comprehensiveness":jg["comprehensiveness"],
            "flat_faithfulness":jf["faithfulness"],
            "graph_faithfulness":jg["faithfulness"],
            "flat_multi_hop_reasoning":jf["multi_hop_reasoning"],
            "graph_multi_hop_reasoning":jg["multi_hop_reasoning"],
            "flat_latency_s":flat["latency_s"],
            "graph_latency_s":graph["latency_s"],
            "flat_total_tokens":flat.get("total_tokens"),
            "graph_total_tokens":graph.get("total_tokens"),
            "flat_judge_rationale":jf["rationale"],
            "graph_judge_rationale":jg["rationale"],
            "graph_supernode_events":len(
                graph["graph_debug"]["diagnostics"].get("supernode_events",[])
            )
        })
        pd.DataFrame(rows).to_csv(CHECKPOINT, index=False)

    out = pd.DataFrame(rows)
    if not out.empty:
        order = {str(x): i for i, x in enumerate(golden_df["id"])}
        out["_order"] = out["id"].astype(str).map(order)
        out = out.sort_values("_order").drop(columns="_order").reset_index(drop=True)
    return out

validate_golden(golden_df, require_answers=True, require_verified=False)
eval_results_df = run_evaluation(golden_df, resume=True)
display(eval_results_df)


⚠️ Golden rows are structurally complete but not all manually verified.
✅ Golden Dataset valid.
Resuming from 1 valid checkpoint rows.


Evaluation:   0%|          | 0/4 [00:00<?, ?it/s]

,id,group,question,reference_answer,flat_answer,graph_answer,flat_comprehensiveness,graph_comprehensiveness,flat_faithfulness,graph_faithfulness,flat_multi_hop_reasoning,graph_multi_hop_reasoning,flat_latency_s,graph_latency_s,flat_total_tokens,graph_total_tokens,flat_judge_rationale,graph_judge_rationale,graph_supernode_events
0,G01,factoid,"According to the corpus, which entity is the target of Noah Kerner -LEADS-> ?",Acorns,"The entity targeted by Noah Kerner's ""LEADS->"" is **Acorns**, as he is identified as its CEO in the context [chunk_i...",The target entity of Noah Kerner -LEADS-> is **Acorns** [chunk_id=0d2f4832beab70ebd1e9]. The context states that Noa...,1,1,1,1,1,1,2.053771,1.948015,985,779,"The candidate context explicitly identifies Noah Kerner as the CEO of Acorns in chunk_id=0d2f4832beab70ebd1e9, direc...","The candidate answer correctly identifies Acorns as the target entity of Noah Kerner -LEADS->, directly supported by...",0
1,G02,multi-hop,"Starting from Sheetal Elangovan, follow two explicit graph relations. Which intermediate entity and final entity are...",Sheetal Elangovan -WORKED_AT-> 1Kosmos; 1Kosmos -DEVELOPED-> BlockID Platform.,"Starting from Sheetal Elangovan, the two explicit graph relations are: \n1. **Sheetal Elangovan → 1Kosmos** (she is...","Starting from Sheetal Elangovan, the two explicit graph relations are: \n1. **Sheetal Elangovan → 1Kosmos** (she is...",1,1,1,1,1,1,3.754500,2.793124,1299,929,The candidate answer correctly identifies the intermediate entity (1Kosmos) and final entity (BlockID platform) base...,The candidate answer correctly identifies the two explicit graph relations (Sheetal Elangovan → 1Kosmos and 1Kosmos ...,0
2,G03,cross-doc,"Using evidence from at least two chunks, summarize two documented relations involving Microsoft and place them in ch...","1) Microsoft acquired Activision Blizzard (date=2023-01-27, chunk=344312edbb3c92e1e2c8::c0000); 2) Microsoft acquire...",Microsoft's documented relations include: \n1. **Microsoft and ISRO (Indian Space Research Organisation):** On Janu...,"1. **Microsoft ACQUIRED Activision Blizzard** on **2023-01-27** [chunk=344312edbb3c92e1e2c8::c0000], as documented i...",1,1,1,1,1,1,3.614734,13.840468,1290,3719,"The candidate answer incorrectly states the ISRO MoU occurred on January 5, 2023, but the candidate context chunk (9...",The candidate answer incorrectly substitutes an AI chip development (2023-10-08) for the second documented Microsoft...,0
3,G04,multi-hop,"Which shared entity is connected to both Gatzby and Databricks Lakehouse for Manufacturing, and what are the two rel...",The shared entity is AI. Gatzby -USES-> AI; Databricks Lakehouse for Manufacturing -USES-> AI.,The shared entity connected to both Gatzby and Databricks Lakehouse for Manufacturing is **the partnership** between...,The shared entity connected to both Gatzby and Databricks Lakehouse for Manufacturing is **the partnership** between...,1,1,1,1,1,1,3.963536,5.857511,1330,1504,"The candidate incorrectly identifies 'the partnership' as the shared entity, which contradicts the reference answer ...","The candidate incorrectly identifies 'the partnership' as the shared entity, which conflates the companies' relation...",0
4,G05,cross-doc,The corpus mentions the relation Microsoft -ACQUIRED-> Activision Blizzard in multiple chunks. What evidence appears...,"Microsoft acquired Activision Blizzard (date=2023-01-27, chunk=344312edbb3c92e1e2c8::c0000); Microsoft acquired Acti...",The corpus mentions the Microsoft-Activision Blizzard acquisition in multiple chunks. Two separate chunks with evide...,The corpus mentions the Microsoft-Activision Blizzard acquisition in four chunks with distinct dates and evidence:\n...,1,5,1,5,1,5,5.435748,7.025585,1693,2468,The candidate answer cites incorrect chunk IDs and dates compared to the reference. The reference specifies chunks 3...,"The candidate answer correctly identifies four chunks with the Microsoft-Activision Blizzard acqu

In [43]:
#@title 4.4 — Comparison table + export
def comparison_table(eval_df):
    metric_map = {
        "Comprehensiveness":("flat_comprehensiveness","graph_comprehensiveness"),
        "Faithfulness":("flat_faithfulness","graph_faithfulness"),
        "Multi-hop reasoning":("flat_multi_hop_reasoning","graph_multi_hop_reasoning"),
        "Latency (s)":("flat_latency_s","graph_latency_s"),
        "Token usage":("flat_total_tokens","graph_total_tokens"),
    }

    rows = []
    for group, g in eval_df.groupby("group"):
        for metric, (fc,gc) in metric_map.items():
            f = pd.to_numeric(g[fc], errors="coerce").mean()
            gr = pd.to_numeric(g[gc], errors="coerce").mean()

            if metric in {"Latency (s)","Token usage"}:
                comment = "Flat RAG thường rẻ/nhanh hơn." if f < gr else "GraphRAG không đắt hơn trong sample này."
            else:
                delta = gr - f
                if delta >= .75:
                    comment = "GraphRAG cải thiện rõ; kiểm tra rationale và provenance."
                elif delta <= -.5:
                    comment = "Flat RAG tốt hơn; graph extraction/retrieval có thể gây mất thông tin hoặc nhiễu."
                else:
                    comment = "Hai phương pháp gần nhau."

            rows.append({
                "Loại câu hỏi":group, "Metric":metric,
                "Flat RAG":round(f,3) if pd.notna(f) else np.nan,
                "GraphRAG":round(gr,3) if pd.notna(gr) else np.nan,
                "Nhận xét phân tích":comment
            })
    return pd.DataFrame(rows)

comparison_df = comparison_table(eval_results_df)
display(comparison_df)
eval_results_df.to_csv("/content/graphrag_eval_results.csv", index=False)
comparison_df.to_csv("/content/graphrag_vs_flatrag_summary.csv", index=False)
print("✅ Exported evaluation CSVs to /content/.")


,Loại câu hỏi,Metric,Flat RAG,GraphRAG,Nhận xét phân tích
0,cross-doc,Comprehensiveness,1.000,3.000,GraphRAG cải thiện rõ; kiểm tra rationale và provenance.
1,cross-doc,Faithfulness,1.000,3.000,GraphRAG cải thiện rõ; kiểm tra rationale và provenance.
2,cross-doc,Multi-hop reasoning,1.000,3.000,GraphRAG cải thiện rõ; kiểm tra rationale và provenance.
3,cross-doc,Latency (s),4.525,10.433,Flat RAG thường rẻ/nhanh hơn.
4,cross-doc,Token usage,1491.500,3093.500,Flat RAG thường rẻ/nhanh hơn.
5,factoid,Comprehensiveness,1.000,1.000,Hai phương pháp gần nhau.
6,factoid,Faithfulness,1.000,1.000,Hai phương pháp gần nhau.
7,factoid,Multi-hop reasoning,1.000,1.000,Hai phương pháp gần nhau.
8,factoid,Latency (s),2.054,1.948,GraphRAG không đắt hơn trong sample này.
9,factoid,Token usage,985.000,779.000,GraphRAG không đắt hơn trong sample này.


✅ Exported evaluation CSVs to /content/.


# PHẦN 5 — FAILURE-MODE CHECKS & SUBMISSION (105–120')

Bắt buộc chứng minh:
1. Edge provenance không thiếu.
2. Entity Resolution có audit.
3. Super-node degree > 100 chỉ expand tối đa 50 edge.
4. Có comparison table.

In [44]:
#@title 5.1 — Super-node check + entity audit
def test_supernode_policy():
    rows = run_cypher("""
    MATCH (n:Entity)-[r]-()
    WITH n, count(r) AS degree
    ORDER BY degree DESC LIMIT 1
    RETURN n.id AS id, n.name AS name, degree
    """)
    if not rows:
        print("Graph empty.")
        return

    n = rows[0]
    limit = 50 if n["degree"] > SUPER_NODE_DEGREE else 1000
    edges = recent_edges(n["id"], limit)
    print(n, "fetched=", len(edges))
    if n["degree"] > SUPER_NODE_DEGREE:
        assert len(edges) <= 50
        print("✅ Super-node cap OK.")

def show_resolution_audit(audit_df):
    if audit_df.empty:
        print("No audit rows.")
        return
    display(
        audit_df.sort_values("similarity", ascending=False).head(30)
    )
    print("High-similarity rejected pairs:")
    display(
        audit_df[audit_df.decision=="REJECT_GUARD"]
        .sort_values("similarity", ascending=False)
        .head(20)
    )

test_supernode_policy()
show_resolution_audit(entity_resolution_audit_df)


{'id': 'fe9221520afe23f76d432625', 'name': 'Databricks Lakehouse for Manufacturing', 'degree': 10} fetched= 10
No audit rows.


## 5.2 — Thuyết minh kỹ thuật

Cell dưới tự tổng hợp các giá trị đo được (threshold, rejected pair, top super-node, nhóm thắng, latency/token)
và ghép với phần giải thích failure modes. Nội dung được xuất ra `/content/graphrag_technical_explanation.md`.

> Nếu Golden Dataset đang dùng `AUTO_FROM_EXTRACTED_EVIDENCE`, kết quả quality chỉ là **bootstrap benchmark**.
> Trước khi coi là kết quả cuối cùng, cần kiểm tra các `reference_evidence` trực tiếp trong raw chunks và đánh dấu `gold_verified=True`.


In [45]:
#@title 5.2B — Auto-generate technical explanation
def _winner_by_group(eval_df):
    if eval_df is None or eval_df.empty:
        return "Evaluation chưa chạy."
    rows = []
    quality_pairs = [
        ("comprehensiveness", "flat_comprehensiveness", "graph_comprehensiveness"),
        ("faithfulness", "flat_faithfulness", "graph_faithfulness"),
        ("multi-hop", "flat_multi_hop_reasoning", "graph_multi_hop_reasoning"),
    ]
    for group, g in eval_df.groupby("group"):
        flat = np.nanmean([
            pd.to_numeric(g[f], errors="coerce").mean()
            for _, f, _ in quality_pairs
        ])
        graph = np.nanmean([
            pd.to_numeric(g[gr], errors="coerce").mean()
            for _, _, gr in quality_pairs
        ])
        winner = "GraphRAG" if graph > flat else ("Flat RAG" if flat > graph else "Tie")
        rows.append(f"- {group}: {winner} (Flat={flat:.2f}, Graph={graph:.2f})")
    return "\n".join(rows)

def build_technical_explanation(
    entity_audit_df,
    top_degree_df,
    eval_df,
    entity_threshold=ENTITY_VECTOR_THRESHOLD,
):
    rejected = pd.DataFrame()
    if entity_audit_df is not None and not entity_audit_df.empty:
        rejected = (
            entity_audit_df[entity_audit_df["decision"] == "REJECT_GUARD"]
            .sort_values("similarity", ascending=False)
        )

    if rejected.empty:
        rejected_text = "Không có REJECT_GUARD trong sample hiện tại."
    else:
        r = rejected.iloc[0]
        rejected_text = (
            f"`{r['left']}` vs `{r['right']}` "
            f"(similarity={float(r['similarity']):.3f}, reason={r.get('guard_reason','')})."
        )

    if top_degree_df is None or top_degree_df.empty:
        top3_text = "Graph chưa có node."
    else:
        top3 = top_degree_df.head(3)
        top3_text = "; ".join(
            f"{r['name']} (degree={int(r['degree'])})"
            for _, r in top3.iterrows()
        )

    if eval_df is None or eval_df.empty:
        latency_text = "Evaluation chưa chạy."
    else:
        fl = pd.to_numeric(eval_df["flat_latency_s"], errors="coerce").mean()
        gl = pd.to_numeric(eval_df["graph_latency_s"], errors="coerce").mean()
        ft = pd.to_numeric(eval_df["flat_total_tokens"], errors="coerce").mean()
        gt = pd.to_numeric(eval_df["graph_total_tokens"], errors="coerce").mean()
        latency_text = (
            f"Mean latency: Flat={fl:.3f}s, Graph={gl:.3f}s. "
            f"Mean tokens: Flat={ft:.1f}, Graph={gt:.1f}. "
            "GraphRAG thường tốn thêm traversal + context assembly, đổi lại có thể tăng coverage cho câu multi-hop."
        )

    winners = _winner_by_group(eval_df)

    md = f'''
# Technical Explanation — GraphRAG vs Flat RAG

1. **Coreference sai ở tình huống nào?**
   Sai nguy hiểm nhất khi một đại từ hoặc mô tả chung có từ hai antecedent hợp lý trong cùng chunk.
   Nếu resolver ép chọn một entity, triple extractor có thể tạo false edge. Pipeline này chỉ resolve khi antecedent rõ;
   trường hợp mơ hồ giữ nguyên và log `unresolved_mentions`.

2. **Entity threshold bao nhiêu, vì sao?**
   Embedding threshold hiện tại là **{entity_threshold:.2f}**. Embedding chỉ tạo candidate; quyết định merge còn qua
   type-aware lexical guard. Company cho phép khác suffix pháp nhân, Person yêu cầu first/last name phù hợp,
   Technology dùng guard chặt hơn để tránh nhập nhầm product.

3. **Candidate similarity cao nhưng không nên merge?**
   {rejected_text}

4. **Top 3 super-node và degree?**
   {top3_text}

5. **Vì sao ưu tiên edge mới nhất có thể đúng/sai?**
   Đúng khi câu hỏi cần trạng thái hiện tại hoặc quan hệ đã thay đổi theo thời gian. Sai khi câu hỏi mang tính lịch sử,
   khi `published_date` thiếu/không chuẩn, hoặc khi bài mới chỉ nhắc lại tin cũ. Vì vậy provenance luôn được giữ để audit.

6–7. **Flat RAG thắng nhóm nào / GraphRAG thắng nhóm nào?**
{winners}

8. **Latency/token trade-off?**
   {latency_text}

9. **AI Coding Agent đề xuất gì mà không dùng, vì sao?**
   Không dùng pairwise cosine toàn bộ document/entity (`O(N²)`) cho near-dedup. Thay vào đó dùng MinHash/LSH để sinh
   candidate rồi exact-Jaccard verify, giúp scale tốt hơn và vẫn có audit table cho false positives.

10. **Scale 350MB: bottleneck đầu tiên là gì?**
    Bottleneck thực tế thường là throughput/cost của coreference + triple extraction qua LLM trước khi Neo4j trở thành
    vấn đề. Cách scale: streaming input, exact/near dedup trước LLM, batching, checkpoint/resume, giới hạn extraction sample,
    và bulk `UNWIND` khi ghi graph.
'''.strip()

    return md

technical_explanation_md = build_technical_explanation(
    entity_resolution_audit_df,
    top_degree_df,
    eval_results_df,
)

print(technical_explanation_md)
Path("/content/graphrag_technical_explanation.md").write_text(
    technical_explanation_md,
    encoding="utf-8"
)
print("✅ Saved /content/graphrag_technical_explanation.md")


# Technical Explanation — GraphRAG vs Flat RAG

1. **Coreference sai ở tình huống nào?**
   Sai nguy hiểm nhất khi một đại từ hoặc mô tả chung có từ hai antecedent hợp lý trong cùng chunk.
   Nếu resolver ép chọn một entity, triple extractor có thể tạo false edge. Pipeline này chỉ resolve khi antecedent rõ;
   trường hợp mơ hồ giữ nguyên và log `unresolved_mentions`.

2. **Entity threshold bao nhiêu, vì sao?**
   Embedding threshold hiện tại là **0.90**. Embedding chỉ tạo candidate; quyết định merge còn qua
   type-aware lexical guard. Company cho phép khác suffix pháp nhân, Person yêu cầu first/last name phù hợp,
   Technology dùng guard chặt hơn để tránh nhập nhầm product.

3. **Candidate similarity cao nhưng không nên merge?**
   Không có REJECT_GUARD trong sample hiện tại.

4. **Top 3 super-node và degree?**
   Apple (degree=10); Databricks Lakehouse for Manufacturing (degree=10); Microsoft (degree=7)

5. **Vì sao ưu tiên edge mới nhất có thể đúng/sai?**
   Đúng khi câu hỏi cần trạ

# 🎁 BONUS

## A — Low-level / High-level
Tạo local entities và high-level topics/community reports; query router chọn tầng retrieval.

## B — Global Search via Community Reports
Nếu Neo4j instance không có GDS phù hợp, fallback:
1. export edges,
2. NetworkX community detection,
3. `UNWIND` write `community_id`,
4. LLM summarize community,
5. query global trên reports.

## C — Self-Correction Graph Retrieval
- hop 2 → LLM kiểm tra context đủ chưa,
- thiếu → hop 3,
- vẫn thiếu → vector fallback,
- bắt buộc stop condition.

In [46]:
#@title Bonus — NetworkX community fallback
import networkx as nx

def build_communities(limit_edges=20000):
    edge_df = pd.DataFrame(run_cypher("""
    MATCH (a:Entity)-[r]->(b:Entity)
    RETURN a.id AS source, b.id AS target
    LIMIT $limit
    """, limit=int(limit_edges)))

    if edge_df.empty:
        print("Graph has no edges; no communities to build.")
        return pd.DataFrame(columns=["id", "community_id"])

    G = nx.Graph()
    G.add_edges_from(edge_df[["source","target"]].itertuples(index=False, name=None))
    communities = nx.algorithms.community.greedy_modularity_communities(G)

    rows = []
    for cid, members in enumerate(communities):
        rows += [{"id":node_id,"community_id":int(cid)} for node_id in members]

    for b in batches(rows, 1000):
        run_cypher("""
        UNWIND $rows AS row
        MATCH (n:Entity {id:row.id})
        SET n.community_id=row.community_id
        """, rows=b)

    return pd.DataFrame(rows)

# community_df = build_communities()


In [47]:
#@title Bonus — Self-correction scaffold
SUFFICIENCY_SYSTEM = """
Decide whether the supplied retrieval context is sufficient to answer the question faithfully.
Do not answer the question. Return strict JSON only.
""".strip()

def context_sufficient(question, context):
    obj, _ = openrouter_json(
        SUFFICIENCY_SYSTEM,
        f"""QUESTION: {question}
CONTEXT:
{context[:16000]}
Return {{"sufficient":true,"missing":"..."}}"""
    )
    return bool(obj.get("sufficient")), norm_space(obj.get("missing"))

def self_correcting_context(question):
    g2 = retrieve_graph_context(question, 2, 100, True)
    ok, missing = context_sufficient(question, g2["context"])
    if ok:
        return {"route":"hop2","context":g2["context"],"missing":""}

    g3 = retrieve_graph_context(question, 3, 100, True)
    ok, missing2 = context_sufficient(question, g3["context"])
    if ok:
        return {"route":"hop3","context":g3["context"],"missing":missing}

    flat, _ = retrieve_flat_context(question, k=8)
    return {
        "route":"hop3+vector",
        "context":f"=== GRAPH ===\n{g3['context']}\n\n=== VECTOR ===\n{flat}",
        "missing":missing2
    }


In [48]:
#@title Final — Submission artifact check
required_outputs = {
    "evaluation_results": "/content/graphrag_eval_results.csv",
    "comparison_summary": "/content/graphrag_vs_flatrag_summary.csv",
    "technical_explanation": "/content/graphrag_technical_explanation.md",
}

print("Submission artifacts:")
for name, p in required_outputs.items():
    print(f"- {name}: {'✅' if Path(p).exists() else '❌'} {p}")

print("\nRuntime checks:")
print("- Neo4j connected:", driver is not None)
print("- chunks:", len(chunks_df))
print("- extracted triples:", len(raw_triples_df))
print("- canonical nodes:", len(nodes_df))
print("- invalid provenance edges:", graph_counts.get("invalid_provenance_edges"))
print("- evaluation rows:", len(eval_results_df))
print("- golden manually verified:",
      bool("gold_verified" in golden_df.columns and golden_df["gold_verified"].fillna(False).astype(bool).all()))


Submission artifacts:
- evaluation_results: ✅ /content/graphrag_eval_results.csv
- comparison_summary: ✅ /content/graphrag_vs_flatrag_summary.csv
- technical_explanation: ✅ /content/graphrag_technical_explanation.md

Runtime checks:
- Neo4j connected: True
- chunks: 1500
- extracted triples: 157
- canonical nodes: 251
- invalid provenance edges: 0
- evaluation rows: 5
- golden manually verified: False


# ✅ RUBRIC

- **30% Chạy được code:** graph nạp thành công, schema đúng, xuất bảng.
- **30% Failure modes:** xử lý ít nhất 2/3 vấn đề Super-node, Entity Resolution, Coreference.
- **20% Evaluation:** chạy hết Golden Dataset, phân tích hợp lý.
- **20% Thuyết minh:** giải thích kiến trúc và cách kiểm soát AI Coding Agent.

## Submission checklist
- [ ] Neo4j connected
- [ ] Dedup/chunking đã chạy
- [ ] Coreference spot-check
- [ ] Entity resolution audit
- [ ] `UNWIND` bulk insert
- [ ] 0 edge thiếu provenance
- [ ] Flat RAG chạy
- [ ] GraphRAG chạy
- [ ] Super-node check
- [ ] Golden Dataset có gold answers thật
- [ ] Evaluation chạy hết
- [ ] Export results + summary CSV
- [ ] Thuyết minh kỹ thuật
- [ ] Bonus (nếu có) có định lượng trước/sau